In [2]:
import pandas as pd
import numpy as np
import polars as pl

In [5]:
df_photo = pd.read_csv('../Data/001/auto_photo_categorization.csv', parse_dates=['categorization_date'])
df_photo.head()

,photo_id,user_id,categorization_date
0,1,1,2024-01-03
1,2,2,2024-01-15
2,3,3,2024-01-20
3,4,4,2024-02-05
4,5,5,2024-02-10


In [20]:
pl_photo = pl.read_csv('../Data/001/auto_photo_categorization.csv', try_parse_dates=True)
pl_photo.head()

photo_id,user_id,categorization_date
i64,i64,date
1,1,2024-01-03
2,2,2024-01-15
3,3,2024-01-20
4,4,2024-02-05
5,5,2024-02-10


# Pregunta 1

### Necesitamos medir el nivel de interacción inicial de los usuarios con las funciones de categorización. ¿Cuántas fotos han sido categorizadas por el sistema en enero de 2024?

```SQL
SELECT
    COUNT(*)
FROM automatic_photo_categorization
--WHERE categorization_date BETWEEN '2024-01-01' AND '2024-01-31';
WHERE EXTRACT(MONTH FROM categorization_date) = 1;
```

In [9]:
enero = df_photo['categorization_date'].between('2024-01-01','2024-01-31')

res = df_photo[enero].shape[0]

res

12

In [27]:


# Filtramos usando objetos date de Python
res = pl_photo.filter(
    pl.col('categorization_date').is_between(date(2024, 1, 1), date(2024, 1, 31))
).height

res

12

# Pregunta 2

### ¿Cuál es el número total de usuarios únicos que han interactuado con la función de categorización en febrero de 2024?

```SQL
SELECT
    COUNT(DISTINCT user_id)
FROM automatic_photo_categorization
WHERE categorization_date BETWEEN '2024-02-01' AND '2024-02-28'
```

In [31]:


feb = df_photo[
    (df_photo['categorization_date'].between('2024-02-01','2024-02-29'))
]

res = len(feb['user_id'].unique())

res

10

In [35]:
feb = pl_photo.filter(
    pl.col('categorization_date').is_between(date(2024,2,1),date(2024,2,29))
).select(
    pl.col('user_id').unique()
)

res = len(feb)

res

10

# Pregutna 3

### Para marzo de 2024, calcula el número total de fotos categorizadas por usuario y renombra la columna resultante como total_categorized_photos. Queremos identificar a los usuarios más activos con fines de investigación de usuarios.

```SQL
SELECT
    user_id,
    COUNT(*) AS total_categorized_photos
FROM automatic_photo_categorization
WHERE categorization_date BETWEEN '2024-03-01' AND '2024-03-31'
GROUP BY user_id
ORDER BY total_categorized_photos DESC;
```

In [40]:
marzo = df_photo[
    df_photo['categorization_date'].between('2024-03-01','2024-03-31')
]

res = (
    marzo.groupby('user_id')['photo_id']
    .count()
    .reset_index(name='total_categorized_photos')
    .sort_values(by='total_categorized_photos', ascending=False)
)

res

,user_id,total_categorized_photos
4,5,3
0,1,2
3,4,2
1,2,2
7,10,2
2,3,1
5,7,1
6,8,1


In [44]:
res = pl_photo.filter(
    pl.col('categorization_date').is_between(date(2024,3,1), date(2024,3,31))
).group_by('user_id').agg(
    pl.len().alias('total_categorized_photos')
).sort('total_categorized_photos', descending=True)

res

user_id,total_categorized_photos
i64,u32
5,3
2,2
4,2
1,2
10,2
8,1
7,1
3,1
